In [12]:
import pandas as pd

In [13]:
# Haal gegevens op
ct_eikenboom = pd.read_csv('../zorgdata/gci_clienten_eikenboom.csv', index_col=False)
ct_appelboom = pd.read_csv('../zorgdata/gci_clienten_appelboom.csv', index_col=False)
ct_notenboom = pd.read_csv('../zorgdata/gci_clienten_notenboom.csv', index_col=False)
ct_perenboom = pd.read_csv('../zorgdata/gci_clienten_perenboom.csv', index_col=False)

rp_eikenboom = pd.read_csv('../zorgdata/gci_rapportages_eikenboom.csv', index_col=False)
rp_appelboom = pd.read_csv('../zorgdata/gci_rapportages_appelboom.csv', index_col=False)
rp_notenboom = pd.read_csv('../zorgdata/gci_rapportages_notenboom.csv', index_col=False)
rp_perenboom = pd.read_csv('../zorgdata/gci_rapportages_perenboom.csv', index_col=False)

In [14]:
# Hernoemen van 'kamernummer' naar 'client_id' in de betreffende DataFrames
ct_perenboom.rename(columns={'kamernummer': 'client_id'}, inplace=True)
ct_appelboom.rename(columns={'kamernummer': 'client_id'}, inplace=True)

# Voeg de afdeling toe om een unieke afdeling / client_id combinatie te vormen
ct_eikenboom['afdeling']='eikenboom'
ct_perenboom['afdeling']='perenboom'
ct_appelboom['afdeling']='appelboom'
ct_notenboom['afdeling']='notenboom'

# Voeg samen
df_clienten = pd.concat([ct_eikenboom, ct_appelboom, ct_notenboom, ct_perenboom], ignore_index=True)
df_clienten = df_clienten[['afdeling', 'client_id', 'naam', 'geslacht', 'profiel']]

In [15]:
# Voeg afdeling weer toe
rp_eikenboom['afdeling']='eikenboom'

# Appelboom heeft nog wat afwijkende namen
rp_appelboom.rename(columns={'cliënt_id': 'client_id'}, inplace=True)
rp_appelboom.rename(columns={'week': 'weekno'}, inplace=True)
rp_appelboom['afdeling']='appelboom'

# Fix notenboom. Bij Notenboom ben ik vergeten het weeknummer toe te voegen. 
# Maak een tijdelijke kolom die een combinatie van 'client_id' en 'dag' bevat
rp_notenboom['client_dag'] = rp_notenboom['client_id'] + "_" + rp_notenboom['dag']
# Tel het aantal voorkomens van elke unieke 'client_dag' combinatie cumulatief
rp_notenboom['weekno'] = rp_notenboom.groupby('client_dag').cumcount() + 1
# Verwijder de tijdelijke kolom 'client_dag'
rp_notenboom.drop(columns='client_dag', inplace=True)
rp_notenboom['afdeling']='notenboom'

# Perenboom is als appelboom
rp_perenboom.rename(columns={'cliënt_id': 'client_id'}, inplace=True)
rp_perenboom.rename(columns={'week': 'weekno'}, inplace=True)
rp_perenboom['afdeling']='perenboom'

# Voeg samen
df_rapportages = pd.concat([rp_eikenboom, rp_appelboom, rp_notenboom, rp_perenboom], ignore_index=True)
df_rapportages = df_rapportages[['afdeling', 'client_id', 'weekno', 'dag', 'niveau', 'rapportage', 'onrustscore']]

In [16]:
df_clienten.to_csv('../zorgdata/df_clienten.csv', index=False)
df_rapportages.to_csv('../zorgdata/df_rapportages.csv', index=False)

In [17]:
df_clienten['geslacht'].value_counts()

geslacht
man         60
vrouw       26
onbekend    10
Name: count, dtype: int64

In [18]:
df_clienten.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   afdeling   96 non-null     object
 1   client_id  96 non-null     object
 2   naam       96 non-null     object
 3   geslacht   96 non-null     object
 4   profiel    96 non-null     object
dtypes: object(5)
memory usage: 3.9+ KB


In [19]:
df_rapportages.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6341 entries, 0 to 6340
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   afdeling     6341 non-null   object 
 1   client_id    6341 non-null   object 
 2   weekno       6341 non-null   int64  
 3   dag          6341 non-null   object 
 4   niveau       6341 non-null   object 
 5   rapportage   6341 non-null   object 
 6   onrustscore  6334 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 346.9+ KB


In [20]:
# Geef de stats van de onrustscore per rapportage
df_rapportages['onrustscore'].describe()

count    6334.000000
mean       36.650931
std        21.987785
min         0.000000
25%        15.000000
50%        35.000000
75%        55.000000
max        95.000000
Name: onrustscore, dtype: float64

In [21]:
# En als we de gemiddelde onrustscore per client willen bekijken, zijn we ook geinteresseerd in de statistieken.
df_rapportages.groupby(['afdeling', 'client_id'])['onrustscore'].mean().reset_index().describe()

,onrustscore
count,96.000000
mean,36.929852
std,5.784373
min,27.188406
25%,33.310559
50%,35.303151
75%,39.039441
max,56.000000
